In [1]:
import re
import requests
import string
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from transformers import GPT2Model, GPT2Tokenizer, BertTokenizer, BertModel
import torch

# Unembeddings

In [2]:
gpt2_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
gpt2_model = GPT2Model.from_pretrained('gpt2')
gpt2_wte = gpt2_model.wte.weight.detach().numpy()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [3]:
# config returns the configuration of the model, which includes information about the architecture, hyperparameters, 
# and other settings used during training. It is useful for understanding the structure of the model and how it was trained.
gpt2_model.config

GPT2Config {
  "activation_function": "gelu_new",
  "add_cross_attention": false,
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "dtype": "float32",
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "pad_token_id": null,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "tie_word_embeddings": true,
  "transformers_version": "5.5.4",
  "use_cache": true,
  "vocab_size": 50257
}

In [4]:
# properties that we'll user later
print(F"Vocab size: {gpt2_model.config.vocab_size}")
print(f"Embedding size: {gpt2_model.config.n_embd}")
print(f"Size of embeddings matrix: {gpt2_wte.shape}")

Vocab size: 50257
Embedding size: 768
Size of embeddings matrix: (50257, 768)


In [5]:
unembeddings = gpt2_wte.copy().T
# check that we are not modifying the original embeddings matrix, by checking the memory addresses of both matrices
print(id(unembeddings), id(gpt2_wte))

2880340669232 2881837542704


In [6]:
random_unembedding = np.random.randn(gpt2_model.config.n_embd, gpt2_model.config.vocab_size)

word = " California"

token_id = gpt2_tokenizer.encode(word, add_special_tokens=False)[0]
print(f"Token ID for '{word}': {token_id}")

embedding_vector = gpt2_wte[token_id]
print(f"Embedding vector shape for '{word}': {embedding_vector.shape}")
print(f"Embedding vector for '{word}': {embedding_vector[:5]}...")  # Print the first 5 dimensions for brevity

Token ID for ' California': 3442
Embedding vector shape for ' California': (768,)
Embedding vector for ' California': [0.05361769 0.08409867 0.1368634  0.07564577 0.084752  ]...


In [ ]:
# project the embedding vector onto the unembedding space
projected_vector = embedding_vector @ unembeddings
projected_vector_random = embedding_vector @ random_unembedding

print(f"Projected vector shape: {projected_vector.shape}")
print(f"Projected vector (first 5 dimensions): {projected_vector[:5]}...")
print(f"Random projected vector shape: {projected_vector_random.shape}")
print(f"Random projected vector (first 5 dimensions): {projected_vector_random[:5]}...")

# get the argmax of the projected vector to find the most likely token
predicted_token_id = np.argmax(projected_vector)
predicted_token_id_random = np.argmax(projected_vector_random)
print(f"Predicted token ID from unembedding: {predicted_token_id}")
print(f"Predicted token ID from random unembedding: {predicted_token_id_random}")

# decode the predicted token ID back to a word
predicted_word = gpt2_tokenizer.decode(predicted_token_id)
predicted_word_random = gpt2_tokenizer.decode(predicted_token_id_random)
print(f"Predicted word from unembedding: '{predicted_word}'")
print(f"Predicted word from random unembedding: '{predicted_word_random}'")

# even the original word was " California", the predicted word from the unembedding is "California" without the leading space, 
# this is because we are using argmax instead of cosine similarity, which is more sensitive to the magnitude of the projected vector rather than its direction.

# normally the next token in the sequence is selected based on the highest probability (argmax) from the output of the unembedding layer, 
# which is a linear transformation of the embedding vector.

Projected vector shape: (50257,)
Projected vector (first 5 dimensions): [2.1238046 2.4398541 2.7362309 2.6577973 2.2950969]...
Random projected vector shape: (50257,)
Random projected vector (first 5 dimensions): [-1.22775012 -1.67426846 -2.77578445 -1.16803139 -4.73014622]...
Predicted token ID from unembedding: 25284
Predicted token ID from random unembedding: 30682
Predicted word from unembedding: 'California'
Predicted word from random unembedding: ' Shrine'


In [ ]:
# find the top 5 most similar tokens to the original embedding vector using cosine similarity
similarities = cosine_similarity(embedding_vector.reshape(1, -1), gpt2_wte)[0]
top_5_indices = np.argsort(similarities)[-5:][::-1]
print("Top 5 most similar tokens to the original embedding vector:")
for idx in top_5_indices:
    token = gpt2_tokenizer.decode(idx)
    similarity_score = similarities[idx]
    print(f"Token: '{token}', Similarity Score: {similarity_score:.4f}")

# now find the top 5 by argmax/argsort of the projected vector (dot product)
top_5_projected_indices = np.argsort(projected_vector)[-5:][::-1]
print("\nTop 5 tokens by argmax of the projected vector:")
for idx in top_5_projected_indices:
    token = gpt2_tokenizer.decode(idx)
    projected_score = projected_vector[idx]
    print(f"Token: '{token}', Projected Score: {projected_score:.4f}")

Top 5 most similar tokens to the original embedding vector:
Token: ' California', Similarity Score: 1.0000
Token: 'California', Similarity Score: 0.8451
Token: ' Californ', Similarity Score: 0.7146
Token: ' Calif', Similarity Score: 0.6844
Token: ' Nevada', Similarity Score: 0.6470

Top 5 tokens by argmax of the projected vector:
Token: 'California', Projected Score: 10.1364
Token: ' California', Projected Score: 9.6169
Token: ' Californ', Projected Score: 8.8158
Token: 'Calif', Projected Score: 8.0876
Token: ' Calif', Projected Score: 7.7181
